# Notebook 2 — Entraînement du modèle Continual-DistilBERT
**Pipeline Big Data de Monitoring de la Désinformation en Temps Réel**  
KOMOSSI Sosso — Master 2 BIG DATA IA, UCAO-UUT 2025-2026

Ce notebook documente l'entraînement du modèle DistilBERT multilingue :
- Base : `distilbert-base-multilingual-cased`
- Fine-tuning sur le corpus fusionné (320k articles)
- Early stopping à F1 ≥ 0.93 — résultat obtenu : **Val F1 = 98.49%, AUC = 99.89%**
- Export ONNX INT8 : 75% de compression, latence ~5-6 ms/article


In [ ]:
import json, os, time
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

MODEL_DIR = '../models/pretrained'
ONNX_DIR  = '../models/onnx'
plt.rcParams['figure.figsize'] = (12, 5)
print('Répertoires :')
for d in [MODEL_DIR, ONNX_DIR]:
    print(f'  {d} : {"OK" if os.path.exists(d) else "MANQUANT"}')


## 1. Historique d'entraînement

In [ ]:
history_path = f'{MODEL_DIR}/training_history.json'
if os.path.exists(history_path):
    history = json.load(open(history_path))
    df_hist = pd.DataFrame(history)
    print(df_hist.to_string(index=False))

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(df_hist["epoch"], df_hist["train_f1"], "b-o", label="Train F1", markersize=6)
    ax.plot(df_hist["epoch"], df_hist["val_f1"],   "r-o", label="Val F1",   markersize=6)
    ax.axhline(0.93, color="gray", linestyle="--", label="Seuil early stopping")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("F1 Score (macro)")
    ax.set_title("Courbes d'entraînement — Continual-DistilBERT")
    ax.legend(); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig("../data/processed/training_curves.png", dpi=150, bbox_inches="tight")
    plt.show()
    best = df_hist.loc[df_hist["val_f1"].idxmax()]
    print(f'Meilleure epoch : {best["epoch"]} | Val F1 : {best["val_f1"]:.4f}')
else:
    print("Historique non trouvé — lancer : python scripts/train_model.py")


## 2. Test de latence ONNX INT8

In [ ]:
onnx_file = f'{ONNX_DIR}/model_quantized.onnx'
if os.path.exists(onnx_file):
    from transformers import DistilBertTokenizerFast
    from onnxruntime import InferenceSession
    import torch

    tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_DIR)
    session   = InferenceSession(onnx_file, providers=["CPUExecutionProvider"])
    test_texts = [
        "BREAKING: Scientists discover shocking truth about vaccines!",
        "Reuters: Central bank raises interest rates by 25 basis points.",
        "SHOCKING: Government hides alien contact since 1947",
        "AFP: Leaders summit ends with agreement on climate targets.",
    ]

    print("Test d'inférence ONNX INT8 :")
    latencies = []
    for text in test_texts:
        enc = tokenizer(text, max_length=128, padding="max_length", truncation=True, return_tensors="np")
        t0 = time.perf_counter()
        logits = session.run(None, {
            "input_ids":      enc["input_ids"].astype(np.int64),
            "attention_mask": enc["attention_mask"].astype(np.int64)
        })[0]
        lat = (time.perf_counter() - t0) * 1000
        latencies.append(lat)
        probs = torch.softmax(torch.tensor(logits), dim=-1).squeeze()
        label = "FAKE" if probs.argmax().item() == 1 else "RÉEL"
        conf  = float(probs.max().item())
        print(f"  [{label}] conf={conf:.3f} | {lat:.1f}ms | "{text[:55]}..."")

    print(f"
Latence moyenne : {sum(latencies)/len(latencies):.1f} ms/article (objectif : < 10 ms)")
else:
    print("Modèle ONNX non trouvé — lancer : python scripts/export_onnx.py")
